### Import modules, configure parameters, set static data

In [11]:
import random
from datetime import datetime, timedelta
import pandas as pd
import ollama
import time

OLLAMA_MODEL = "gemma3:latest"

START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2025, 12, 31)

MIN_REVIEWS_PER_TRAIL = 5
MAX_REVIEWS_PER_TRAIL = 15

In [22]:
TRAILS = [
    {
        "name": "Silver Star Mountain via South Ridge",
        "state": "Washington",
        "lat": 45.7820,
        "lon": -122.2920,
        "difficulty": 4,
        "elevation_gain_ft": 3238,
        "peak_elevation_ft": 4364,
        "mileage": 9.8,
        "landmarks": [
            "bear grass",
            "summit views",
            "rocky trail",
            "mountains in distance",
            "boulder field"
            
        ],
        "wildlife": ["deer", "hawk", "chipmunk", "rabbit"]
    },

    {
        "name": "McNeil Point",
        "state": "Oregon",
        "lat": 45.3733,
        "lon": -121.7356,
        "difficulty": 4,
        "elevation_gain_ft": 3375,
        "peak_elevation_ft": 6812,
        "mileage": 9.6,
        "landmarks": [
            "McNeil Point shelter",
            "Mt Hood views",
            "alpine meadows"
        ],
        "wildlife": ["deer", "elk", "hawk"]
    },

    {
        "name": "Angel's Rest to Devil's Rest Loop",
        "state": "Oregon",
        "lat": 45.5605,
        "lon": -122.1720,
        "difficulty": 3,
        "elevation_gain_ft": 3159,
        "peak_elevation_ft": 2450,
        "mileage": 12.0,
        "landmarks": [
            "Angel's Rest viewpoint",
            "Devil's Rest",
            "Columbia Gorge views"
        ],
        "wildlife": ["deer", "chipmunk", "hawk"]
    },

    {
        "name": "Nesmith Point Trail",
        "state": "Oregon",
        "lat": 45.6815,
        "lon": -121.9020,
        "difficulty": 4,
        "elevation_gain_ft": 3746,
        "peak_elevation_ft": 3848,
        "mileage": 9.1,
        "landmarks": [
            "long climb",
            "ridge views",
            "forest sections",
            "steep sections"
        ],
        "wildlife": ["deer", "elk", "rabbit"]
    },

    {
        "name": "Mt Defiance via Starvation Creek Trail",
        "state": "Oregon",
        "lat": 45.6900,
        "lon": -121.7400,
        "difficulty": 5,
        "elevation_gain_ft": 5177,
        "peak_elevation_ft": 5000,
        "mileage": 12.5,
        "landmarks": [
            "Starvation Creek",
            "waterfalls",
            "summit ridge",
            "steep sections"
        ],
        "wildlife": ["deer", "elk", "hawk"]
    },

    {
        "name": "Larch Mountain",
        "state": "Oregon",
        "lat": 45.5396,
        "lon": -122.0866,
        "difficulty": 3,
        "elevation_gain_ft": 4035,
        "peak_elevation_ft": 4061,
        "mileage": 13.5,
        "landmarks": [
            "Sherrard Point",
            "viewpoint",
            "forest trail",
            "steep sections",
            "long trail"
        ],
        "wildlife": ["deer", "chipmunk"]
    },

    {
        "name": "Mt. St Helens Summit via Ptarmigan Trail",
        "state": "Washington",
        "lat": 46.2010,
        "lon": -122.1880,
        "difficulty": 5,
        "elevation_gain_ft": 4639,
        "peak_elevation_ft": 8363,
        "mileage": 8.8,
        "landmarks": [
            "boulder fields",
            "summit crater",
            "volcanic landscape"
        ],
        "wildlife": ["mountain goat", "ptarmigan"]
    },

    {
        "name": "Dog Mountain",
        "state": "Washington",
        "lat": 45.7087,
        "lon": -121.7113,
        "difficulty": 4,
        "elevation_gain_ft": 2936,
        "peak_elevation_ft": 2948,
        "mileage": 7.2,
        "landmarks": [
            "wildflower slopes",
            "summit views",
            "Columbia Gorge",
            "steep sections"
        ],
        "wildlife": ["hawk", "deer", "chipmunk"]
    },

    {
        "name": "Hamilton Mountain",
        "state": "Washington",
        "lat": 45.6654,
        "lon": -121.9648,
        "difficulty": 3,
        "elevation_gain_ft": 2237,
        "peak_elevation_ft": 2438,
        "mileage": 7.5,
        "landmarks": [
            "waterfalls",
            "pool of the winds",
            "ridge views"
        ],
        "wildlife": ["deer", "hawk"]
    },

    {
        "name": "Ramona Falls",
        "state": "Oregon",
        "lat": 45.3738,
        "lon": -121.8269,
        "difficulty": 2,
        "elevation_gain_ft": 1079,
        "peak_elevation_ft": 3560,
        "mileage": 7.2,
        "landmarks": [
            "Ramona Falls",
            "river crossing",
            "forest trail",
            "Mt. Hood"
        ],
        "wildlife": ["deer", "bird", "salamander"]
    }
]

### Define functions that assign probabilities to conditions/animal spotting

In [13]:
def weather_for_month(month):

    r = random.random() 

    if month in [6, 7, 8]:
        if r < 0.70:
            return "sunny"
        elif r >= 0.70 and r < 0.85:
            return "partly cloudy"
        elif r >= 0.85 and r < 0.95:
            return "cloudy"
        else:
            return "rain"

    elif month in [9, 10]:
        if r < 0.40:
            return "sunny"
        elif r >= 0.40 and r < 0.60:
            return "partly cloudy"
        elif r >= 0.60 and r < 0.80:
            return "cloudy"
        elif r >= 0.80 and r < 0.95:
            return "rain"
        else:
            return "foggy"

    elif month in [11, 12, 1, 2]:
        if r < 0.40:
            return "cloudy"
        elif r >= 0.40 and r < 0.75:
            return "rain"
        elif r >= 0.75 and r < 0.90:
            return "foggy"
        else:
            return "sunny"

    else:
        if r < 0.30:
            return "sunny"
        elif r >= 0.30 and r < 0.50:
            return "partly cloudy"
        elif r >= 0.50 and r < 0.75:
            return "cloudy"
        elif r >= 0.75 and r < 0.95:
            return "rain"
        else:
            return "foggy"


def snow_probability(peak_elevation_ft, month):

    if peak_elevation_ft >= 7000:

        if month in [1, 2]:
            return 0.95
        elif month in [3,4]:
            return 0.85
        elif month in [5,6]:
            return 0.75
        elif month in [7,8]:
            return 0.10
        elif month == 9:
            return 0.05
        elif month == 10:
            return 0.30
        elif month == 11:
            return 0.75
        else:
            return 0.95


    elif peak_elevation_ft > 4500 and peak_elevation_ft < 7000:

        if month in [1, 2]:
            return 0.90
        elif month in [3,4]:
            return 0.70
        elif month in [5,6]:
            return 0.30
        elif month == 7:
            return 0.02
        elif month in [8, 9]:
            return 0.00
        elif month == 10:
            return 0.10
        elif month == 11:
            return 0.40
        else:
            return 0.75


    else:

        if month in [1,2]:
            return 0.25
        elif month == 3:
            return 0.10
        elif month in [4,5]:
            return 0.02
        elif month in [6, 7, 8, 9]:
            return 0.00
        elif month == 10:
            return 0.01
        elif month == 11:
            return 0.05
        else:
            return 0.15


def snow_present(peak_elevation_ft, month):

    chance = snow_probability(peak_elevation_ft, month)

    choices = [True, False]

    return random.choices(
        choices,
        weights=[chance, 1 - chance],
        k=1
    )[0]



def overgrowth_probability(review_count):

    if review_count < 125:
        return 0.30

    elif review_count >= 125 and review_count < 150:
        return 0.25

    elif review_count >= 150 and review_count < 175:
        return 0.20

    elif review_count >= 175:
        return 0.15

    else:
        return 0.10


def overgrowth_present(review_count):

    chance = overgrowth_probability(review_count)
    choices = [True, False]

    return random.choices(
        choices,        
        weights=[chance, 1 - chance],
        k=1
    )[0]


def bear_probability(month):

    if month in [6, 7, 8]:
        return 0.025

    if month in [4, 5, 9, 10]:
        return 0.015

    return 0.002


def wildlife_seen(trail, month):

    if month in [6, 7, 8]:
        chance = 0.50

    elif month in [4, 5, 9, 10]:
        chance = 0.35

    else:
        chance = 0.20


### Define function for generate context in prompts

In [14]:
def generate_context(trail, review_date, review_count):

    month = review_date.month

    return {
        "weather": weather_for_month(month),
        "snow_present": snow_present(trail["peak_elevation_ft"], month),
        "wildlife": wildlife_seen(trail, month),
        "overgrowth": overgrowth_present(review_count),
        "landmark": random.choice(trail["landmarks"])
    }

### Define function for generating rating

In [15]:
def generate_rating(trail, context):

    rating = 4.5 - (trail["difficulty"] * 0.15)

    if context["snow_present"]:
        rating -= 0.2

    if context["overgrowth"]:
        rating -= 0.4

    rating = random.gauss(rating, 0.8)
    rating = min(5, max(1, rating))

    return round(rating)

### Let's put it all together and construct the prompts

In [16]:
system_prompt = f"""

    You are an experienced hiker writing realistic trail reviews
    similar to reviews found on AllTrails.
    
    Requirements:
    - Write 1 to 5 sentences.
    - Sound like a real hiker.
    - Usually positive but not overly enthusiastic.
    - Mention trail conditions when relevant.
    - Mention trail parameters (eg: elevation and mileage) if relevant, but estimate it; does not need to be exact.
    - Sometimes use imperfect grammar or punctuation.
    - Each review needs to have an unique voice and be different from all the others.
    - Do not start the review with "Okay, wow" or "Okay," or "Okay, heres a realistic hiking review for..."
    - Do not mention every piece of information provided.
    - Do not cite major events that were not supplied.
    - No bullet points.
    - No headings.
    - No explanations.
    - Output only the review text.

    """

In [17]:
def generate_review_text(
    trail,
    review_date,
    rating,
    context
):
    user_prompt = f"""

    Write a realistic hiking review similar to what would
    appear on AllTrails using the information below:
    
    
    Trail:
    {trail["name"]}
    
    Date:
    {review_date.strftime("%Y-%m-%d")}
    
    Difficulty:
    {trail["difficulty"]}/5
    
    Elevation Gain:
    {trail["elevation_gain_ft"]} ft
    
    Rating:
    {rating}/5
    
    Conditions:
    Weather={context["weather"]}
    Snow={context["snow_present"]}
    Wildlife={context["wildlife"]}
    Overgrowth={context["overgrowth"]}
    Landmark={context["landmark"]}
    
    Write only the review.

    """

    
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }]
    )

    return (response["message"]["content"].strip())

In [18]:
def random_date():

    total_days = (END_DATE - START_DATE).days
    return START_DATE + timedelta(days=random.randint(0, total_days))

In [19]:
def generate_dataset():

    rows = []
    review_id = 1

    for trail in TRAILS:

        review_count = random.randint(
            MIN_REVIEWS_PER_TRAIL,
            MAX_REVIEWS_PER_TRAIL
        )

        print(
            f"Generating {review_count} reviews "
            f"for {trail['name']}"
        )

        trail_start = time.perf_counter()

        for review_num in range(review_count):

            review_date = random_date()

            context = generate_context(
                trail,
                review_date,
                review_count
            )

            rating = generate_rating(
                trail,
                context
            )

            review_text = generate_review_text(
                trail,
                review_date,
                rating,
                context
            )

            rows.append({
                "review_id": review_id,
                "trail_name": trail["name"],
                "state": trail["state"],
                "latitude": trail["lat"],
                "longitude": trail["lon"],
                "difficulty": trail["difficulty"],
                "elevation_gain_ft": trail["elevation_gain_ft"],
                "rating": rating,
                "date": review_date.date().isoformat(),
                "review_text": review_text
            })

            review_id += 1

        trail_elapsed = time.perf_counter() - trail_start

        print(
            f"Completed {trail['name']} | "
            f"{trail_elapsed:.1f}s"
        )

    return pd.DataFrame(rows)

In [20]:
df = generate_dataset()
df.to_csv("synthetic_hiking_reviews.csv", index=False)

print("\nDataset complete.")
print("Rows:", len(df))
df.head()

Generating 11 reviews for Silver Star Mountain via South Ridge
Completed Silver Star Mountain via South Ridge | 54.5s
Generating 5 reviews for McNeil Point
Completed McNeil Point | 22.2s
Generating 15 reviews for Angel's Rest to Devil's Rest Loop
Completed Angel's Rest to Devil's Rest Loop | 68.4s
Generating 10 reviews for Nesmith Point Trail
Completed Nesmith Point Trail | 42.5s
Generating 15 reviews for Mt Defiance via Starvation Creek Trail
Completed Mt Defiance via Starvation Creek Trail | 68.0s
Generating 10 reviews for Larch Mountain
Completed Larch Mountain | 40.3s
Generating 5 reviews for Mt. St Helens Summit via Ptarmigan Trail
Completed Mt. St Helens Summit via Ptarmigan Trail | 22.2s
Generating 14 reviews for Dog Mountain
Completed Dog Mountain | 61.9s
Generating 5 reviews for Hamilton Mountain
Completed Hamilton Mountain | 20.8s
Generating 11 reviews for Ramona Falls
Completed Ramona Falls | 46.3s

Dataset complete.
Rows: 101


,review_id,trail_name,state,latitude,longitude,difficulty,elevation_gain_ft,rating,date,review_text
0,1,Silver Star Mountain via South Ridge,Washington,45.782,-122.292,4,3238,5,2022-09-14,"This was a tough one, no doubt about it – that..."
1,2,Silver Star Mountain via South Ridge,Washington,45.782,-122.292,4,3238,3,2022-10-18,"This was a tough one, no doubt about it. The S..."
2,3,Silver Star Mountain via South Ridge,Washington,45.782,-122.292,4,3238,4,2021-04-23,"This was a solid climb, no doubt about it. The..."
3,4,Silver Star Mountain via South Ridge,Washington,45.782,-122.292,4,3238,4,2021-06-04,"June 4th, 2021 – This one was a burner. The So..."
4,5,Silver Star Mountain via South Ridge,Washington,45.782,-122.292,4,3238,4,2024-07-11,July 11th was a beautiful day for tackling Sil...
